# Document Classification Using NLP Techniques

**Major Project**  
**Author:** Namami Sharma  
**College:** Samrat Ashoka Technological Institute, Vidisha

## Objective
Classify text documents into Sports, Technology, Business, Education, Entertainment, or Politics using NLP and Machine Learning.

## 1. Project Workflow

Text Dataset → Text Cleaning → TF-IDF → Machine Learning → Evaluation → New Document Prediction

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


## 3. Load Dataset

Keep `Document_Classification_NLP_Dataset.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv("Document_Classification_NLP_Dataset.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
print(df["Category"].value_counts())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 4. Text Cleaning

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df = df.dropna(subset=["Document","Category"]).drop_duplicates().reset_index(drop=True)
df["Cleaned_Document"] = df["Document"].apply(clean_text)
df[["Document","Cleaned_Document","Category"]].head()


## 5. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(data=df,x="Category",order=df["Category"].value_counts().index)
plt.title("Documents by Category")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 6. Train/Test Split

In [ ]:
X = df["Cleaned_Document"]
y = df["Category"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print("Training:",len(X_train)," Testing:",len(X_test))


## 7. TF-IDF Feature Extraction

TF-IDF converts text into numerical features. It gives useful words higher importance and reduces the importance of very common words.

In [ ]:
tfidf=TfidfVectorizer(stop_words="english",ngram_range=(1,2),max_features=10000,sublinear_tf=True)
X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)
print("TF-IDF training shape:",X_train_tfidf.shape)


## 8. Train Machine Learning Models

In [ ]:
models={
    "Logistic Regression":LogisticRegression(max_iter=2000,random_state=42),
    "Multinomial Naive Bayes":MultinomialNB(),
    "Linear SVM":LinearSVC(random_state=42)
}
results=[]
trained={}
for name,model in models.items():
    model.fit(X_train_tfidf,y_train)
    pred=model.predict(X_test_tfidf)
    results.append({
        "Model":name,
        "Accuracy":accuracy_score(y_test,pred),
        "Precision":precision_score(y_test,pred,average="weighted",zero_division=0),
        "Recall":recall_score(y_test,pred,average="weighted",zero_division=0),
        "F1 Score":f1_score(y_test,pred,average="weighted",zero_division=0)
    })
    trained[name]=model
results_df=pd.DataFrame(results).sort_values("F1 Score",ascending=False)
results_df


## 9. Model Comparison

In [ ]:
results_df.set_index("Model")[["Accuracy","Precision","Recall","F1 Score"]].plot(kind="bar",figsize=(10,5))
plt.ylim(0,1.05); plt.title("Model Performance Comparison"); plt.ylabel("Score"); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()


## 10. Classification Reports

In [ ]:
for name,model in trained.items():
    pred=model.predict(X_test_tfidf)
    print("="*65)
    print(name)
    print("="*65)
    print(classification_report(y_test,pred,zero_division=0))


## 11. Confusion Matrix

In [ ]:
best_model_name=results_df.iloc[0]["Model"]
best_model=trained[best_model_name]
best_pred=best_model.predict(X_test_tfidf)
labels=sorted(y_test.unique())
cm=confusion_matrix(y_test,best_pred,labels=labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=labels,yticklabels=labels)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - "+best_model_name)
plt.tight_layout(); plt.show()
print("Model selected for demonstration:",best_model_name)


## 12. New Document Prediction

In [ ]:
new_document=["The football team won the championship after a close match."]
vector=tfidf.transform([clean_text(new_document[0])])
prediction=best_model.predict(vector)[0]
print("Document:",new_document[0])
print("Predicted Category:",prediction)


## 13. Try Your Own Text

In [ ]:
user_text=input("Enter a document/text: ")
prediction=best_model.predict(tfidf.transform([clean_text(user_text)]))[0]
print("Predicted Category:",prediction)


## 14. Conclusion

This project demonstrates text cleaning, TF-IDF feature extraction, Machine Learning classification, model comparison, evaluation, confusion matrix visualization, and prediction of new documents.

**Note:** The supplied dataset is synthetic and intended for educational/academic demonstration.